# 튜닝 4-1 — LightGBM 순위 회귀: 깊이 3·4 × 트리 상한 2000 + 누수 점검 (로컬 CPU)

- **조건(깊이 3·4 같음)**: 주 세트(기본+H1+H3+H2+H4, RMW 포함), 학습률 0.01, L2 0, 16 Fold, 내부 검증 직전 1년, H3 λ 3개. 결측 평균 채움 등 전처리는 basefeat 실행과 같다.
- **트리**: 2000개까지 한 번 학습하고 해마다 선택(1~2000)과 500·1000·1500·2000 고정을 같은 학습에서 잘라 비교.
- **누수 점검(깊이 3)**: ① 한 달 늦추기 — 입력을 t-1 값으로 바꿔 t+1 예측. 미래 정보가 섞였다면 IC가 크게 무너진다. ② 정답 섞기 — 같은 달 안에서 목표를 섞어 학습, 원래 수익률로 평가. IC가 0 근처여야 정상.
- **판단**: 같은 트리 기준끼리 깊이 3 vs 4, 0.002 미만이면 단순한 쪽(깊이 3, 트리 적은 쪽).
- **중간 저장**: 변형마다 `outputs/model_results/jiun/tune_lgb_v1_parts/`에 저장, 끊기면 Run All로 이어감. 요약 표는 `outputs/model_results/jiun/tune_lgb_v1_summary.csv` 하나. 예상 약 1~1.5시간.

## 1. 설정

In [ ]:
import os, sys
from datetime import datetime
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from src.evaluation import model_test as mt
from src.evaluation import base_model_runs as bmr
from src.features.h3 import compute_h3_sigma_grid

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
mt.check_no_test(config)

AUTHOR = 'jiun'
RUN_NAME = 'tune_lgb_v1'        # 같은 이름으로 다시 Run All 하면 저장된 단위를 이어 씀
MAX_TREES = 2000                # 트리 상한(기존 1000). 2000까지 한 번 학습하고 아래 고정 개수로 잘라 비교
FIXED_TREES = [500, 1000, 1500, 2000]
LGB_THREADS = None
FOLDS = None                    # None = 16 Fold 전부
SET = 'base+h1+h3+h2+h4'        # 주 세트(RMW 포함)
mt.boosting.MAX_TREES = MAX_TREES   # 노트북 안에서만 상한을 바꾼다(코드 기본값 1000은 그대로)
PARTS = bmr.parts_dir(ROOT, config, author=AUTHOR, run_name=RUN_NAME)
print('중간 저장 폴더:', PARTS, '| 이미 저장된 단위:', len(list(PARTS.glob('*.pkl'))))

## 2. H3 격자·전처리·누수 점검용 자료

In [ ]:
panel = pd.read_parquet(ROOT / config['PATHS']['panel'], columns=['stock_id', 'month', 'return', 'eligible'])
eligibility = pd.read_parquet(ROOT / config['PATHS']['eligibility'])
H3_GRID = compute_h3_sigma_grid(panel, eligibility)
del panel, eligibility
PREP = bmr.prepare(ROOT, config, 'boosting', input_transform='rank', h3_grid=H3_GRID)
COLS = bmr.input_sets('boosting')[SET]
assert 'rmw' in COLS, 'RMW가 없는 피처 표다'
D = PREP['data']
XCOLS = [c for c in D.columns if c.startswith('x_')]
print('입력', len(COLS), '| x 열', len(XCOLS))

# 누수 점검 ① 한 달 늦추기: 모든 입력을 한 달 전 값(t-1)으로 바꿔 같은 목표(t+1)를 예측. 빈 칸은 그 달 평균(변환 입력은 0)
prev = D[['stock_id', 'target_month', *XCOLS]].copy()
prev['target_month'] = (pd.PeriodIndex(prev.target_month, freq='M') + 1).astype(str)
lag = D.drop(columns=XCOLS).merge(prev, on=['stock_id', 'target_month'], how='left')
lag[XCOLS] = lag[XCOLS].fillna(lag.groupby('target_month')[XCOLS].transform('mean')).fillna(0.0)
assert len(lag) == len(D) and (lag.stock_id.to_numpy() == D.stock_id.to_numpy()).all()
PREP_LAG = {**PREP, 'data': lag}

# 누수 점검 ② 정답 섞기: 같은 달 안에서 다음 달 수익률을 종목끼리 무작위로 섞고(seed 0) 목표값을 다시 만든다. 평가는 원래 수익률로
rng = np.random.default_rng(0)
shuf = D.copy()
obs = shuf.y.notna()
shuf.loc[obs, 'y'] = shuf[obs].groupby('target_month').y.transform(lambda s: rng.permutation(s.to_numpy()))
for name, arr in mt.monthly_target_transforms(shuf.y, shuf.train.to_numpy(), shuf.target_month).items():
    shuf['t_' + name] = arr
PREP_SHUF = {**PREP, 'data': shuf}
Y_TRUE = D.set_index(['stock_id', 'target_month']).y

## 3. 실행 (4단위: 깊이 3, 깊이 4, 한 달 늦추기, 정답 섞기)

In [ ]:
folds = None if FOLDS is None else [f for f in mt.plan_folds(config['MODEL_GRID']) if f['fold'] in FOLDS]
VARIANTS = {'d3': (PREP, 3), 'd4': (PREP, 4), 'lag_d3': (PREP_LAG, 3), 'shuffle_d3': (PREP_SHUF, 3)}
fp = bmr.fingerprint(ROOT, config, 'boosting', 'rank')
units = []
for name, (prep, depth) in VARIANTS.items():
    setting = [('lgb_rankreg', 'percentile', 'mse', depth, 0.01)]
    spec = {'set': SET, 'setting': setting, 'variant': name, 'max_trees': MAX_TREES, 'fixed_trees': FIXED_TREES, 'folds': FOLDS}
    units.append((f'tune_lgb__{name}', spec,
                  (lambda prep=prep, setting=setting: mt.run_boosting(prep, config, 'lgb_rankreg', phase='basefeat', settings=setting,
                                                                      sets={SET: COLS}, fixed_trees=FIXED_TREES, folds=folds,
                                                                      n_threads=LGB_THREADS))))
RESULTS = bmr.run_units(units, PARTS, fp)

## 4. 요약

In [ ]:
rows = []
for unit, r in RESULTS.items():
    v = unit.split('__')[1]
    vk = r['val_keys']
    y = Y_TRUE.reindex(pd.MultiIndex.from_frame(vk[['stock_id', 'target_month']])).to_numpy()   # 섞기도 원래 수익률로 평가
    sel = r['selections']
    for key, pred in r['predictions'].items():
        hp = key.split('|')[-1]
        basis = '해마다' if '_t' not in hp else hp.split('_t')[-1] + ' 고정'
        ic = mt.monthly_ic(vk.target_month.to_numpy(), pred, y).ic
        s = sel[sel.hparams == hp]
        rows.append({'변형': v, '트리': basis, '평균 IC': ic.mean(), 'ICIR': ic.mean() / ic.std(ddof=1),
                     '학습 IC': s.train_ic.mean(), '학습−검증': s.train_ic.mean() - ic.mean(),
                     '고른 트리 중앙값': s.trees.median(), f'상한 근처(≥{MAX_TREES - 10})': int((s.trees >= MAX_TREES - 10).sum()),
                     '개월': int(ic.notna().sum())})
T = pd.DataFrame(rows)
order = ['해마다'] + [f'{n} 고정' for n in FIXED_TREES]
T['트리'] = pd.Categorical(T['트리'], order, ordered=True)
T = T.sort_values(['변형', '트리']).reset_index(drop=True)
print(T.round(4).to_string())
out = PARTS.parent / f'{RUN_NAME}_summary.csv'
T.to_csv(out, index=False, encoding='utf-8-sig')
print('저장:', out)